# Study 818 — Trend Factor 🌊

**Does blending *all* the moving-average horizons at once beat any single one — and momentum?**

Han, Zhou & Zhu (2016) build a **trend factor**: for each name form normalized moving-average
signals `A_L = MA_L(price)/price` for `L ∈ {3,5,10,20,50,100,200}`, let a rolling cross-
sectional (Fama-MacBeth) regression *weight* those horizons, and dot the averaged past slopes
into today's signals to get a fitted expected return. Sort **long high-trend / short low-trend**.
The paper's headline is that this blend **beats** single-moving-average timing *and* momentum.
We take the self-contained daily version on a liquid US cross-section (2010-01-04 → 2026-06-30,
50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast synthetic
control. Survivorship: current-membership mega-caps — magnitudes are an upper bound.*


## 1. The idea in one picture

A single moving average looks at *one* time scale. Different traders react over different horizons — days, weeks, months — so a 200-day rule throws away the short-horizon information and vice versa. The trend factor lets a cross-sectional regression *decide the weights* on all seven horizons at once, then buys the names with the highest fitted expected return and sells the lowest.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=1.42, t_nw=0.99, hi_bps=8.48, lo_bps=7.06, gross_sharpe=0.23,
         ma200_bps=2.01, ma200_t=1.24, mom_bps=2.24, mom_t=1.4)
print('long high-trend / short low-trend spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  high-trend book %+.2f bps vs low-trend book %+.2f bps'
      % (R['hi_bps'], R['lo_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

long high-trend / short low-trend spread: +1.42 bps/day (NW t = +0.99)
  high-trend book +8.48 bps vs low-trend book +7.06 bps
  gross spread Sharpe (before cost): 0.23


## 2. Is the machinery even wired right? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: persistent trends that both move the price and predict its next return) and check the fitted trend factor recovers it — and that it stays *silent* on the null (`edge=0`, prices are random walks). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from trend_factor import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=818, n_assets=40, n_days=1500), beta_window=120)
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0015, seed=818, n_assets=40, n_days=1500), beta_window=120)
print('null world   : trend-factor spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: trend-factor spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : trend-factor spread NW t = +0.63  (should be ~0)
planted world: trend-factor spread NW t = +10.28  (should light up)


## 3. The contrast the paper hangs on — and what actually happened

The trend factor is *supposed* to beat single-MA timing and momentum. On 50 liquid US mega-caps it is the **weakest** of the three:

| sort | spread/day | NW *t* |
|---|--:|--:|
| **trend factor (blend of 7)** | **+1.42 bps** | **+0.99** |
| single-MA(200) timing | +2.01 bps | +1.24 |
| 12-1 momentum | +2.24 bps | +1.40 |

All three are insignificant, but the fancy blend adds *nothing* over a plain 200-day rule or vanilla momentum here.

## 4. The honest verdict — the famous factor does *not* replicate here

On this liquid mega-cap tape the long-high-trend / short-low-trend spread is **+1.42 bps/day** with NW *t* = **+0.99** — the right sign but statistically indistinguishable from zero (only ~1.45 sd into a 1,000-permutation placebo, p = 0.066), weak in both eras (*t* = +0.96 / +0.57). The seeded synthetic control recovers a *planted* trend relation cleanly (*t* = +10.28), so this is a genuine null on the mega-cap survivor slice, not a bug — the trend premium is a broad-cross-section / small-cap phenomenon. And the book dies under the lightest costs: at 1 bp one-way, net **-0.72 bps/day**. **Signal: None**, **Tradability: Mirage**.